# 02. Demanda observada en estaciones troncales

**Pregunta principal de esta etapa:** ¿en qué estaciones y horas se registran más entradas al sistema durante los días disponibles?

Este notebook usa **validaciones reales**, no posiciones de buses. Una validación es una entrada registrada, no un pasajero único ni un bus. Los ZIP están en `registros_diarios/AAAA-MM-DD/`, al lado del repositorio. Se procesan por bloques para cuidar la memoria. **Incluye automáticamente todos los ZIP encontrados**: revisa la lista que imprime la primera celda antes de comparar días.

**Límite importante:** este cuaderno describe demanda observada. El proyecto ya preparó cruces posteriores entre validaciones y oferta programada en otros archivos; este cuaderno no los reproduce ni es la fuente activa del tablero. Las validaciones tampoco permiten medir por sí solas cumplimiento o puntualidad de los buses.


## 1. Librerías y ubicación de los archivos

In [ ]:
from pathlib import Path
import pandas as pd
try:
    import matplotlib.pyplot as plt
except ModuleNotFoundError:
    plt = None
    print('Matplotlib no está instalado: se mostrarán las tablas, pero no las gráficas.')

carpeta_proyecto = Path.cwd()
if carpeta_proyecto.name == 'notebooks':
    carpeta_proyecto = carpeta_proyecto.parent

carpeta_registros = carpeta_proyecto.parent / 'registros_diarios'
archivos_zip = sorted(carpeta_registros.glob('*/validacionTroncal*.zip'))

print('Carpeta de registros:', carpeta_registros)
print('Días encontrados:', len(archivos_zip))
for archivo in archivos_zip:
    print(archivo.parent.name, '-', archivo.name)

if len(archivos_zip) == 0:
    raise FileNotFoundError('No hay ZIP de validaciones en registros_diarios/AAAA-MM-DD/')


## 2. Leer únicamente las columnas necesarias

Cada ZIP puede contener cientos de millones de caracteres. `chunksize` permite leerlo por partes. No cargamos números de tarjeta ni saldos: no hacen falta para contar entradas por estación y hora. El archivo usa codificación `latin-1`, comprobada en una muestra de sus nombres.

In [ ]:
columnas = ['Fecha_Transaccion', 'Estacion_Parada', 'Sistema']
resumenes = []
control_archivos = []

for archivo in archivos_zip:
    filas_leidas = 0
    fechas_invalidas = 0
    estaciones_vacias = 0
    primera_hora = None
    ultima_hora = None

    for bloque in pd.read_csv(archivo, compression='zip', encoding='latin-1',
                              usecols=columnas, chunksize=100000, low_memory=False):
        filas_leidas += len(bloque)
        bloque['Fecha_Transaccion'] = pd.to_datetime(bloque['Fecha_Transaccion'], errors='coerce')
        fechas_invalidas += bloque['Fecha_Transaccion'].isna().sum()
        estaciones_vacias += bloque['Estacion_Parada'].isna().sum()
        bloque = bloque.dropna(subset=['Fecha_Transaccion', 'Estacion_Parada'])
        bloque = bloque[bloque['Sistema'] == 'TRONCAL'].copy()

        if len(bloque) > 0:
            minimo = bloque['Fecha_Transaccion'].min()
            maximo = bloque['Fecha_Transaccion'].max()
            if primera_hora is None or minimo < primera_hora:
                primera_hora = minimo
            if ultima_hora is None or maximo > ultima_hora:
                ultima_hora = maximo

            bloque['fecha'] = bloque['Fecha_Transaccion'].dt.date
            bloque['hora'] = bloque['Fecha_Transaccion'].dt.hour
            resumen_bloque = bloque.groupby(['fecha', 'hora', 'Estacion_Parada']).size()
            resumenes.append(resumen_bloque.reset_index(name='validaciones'))

    control_archivos.append([archivo.name, filas_leidas, fechas_invalidas,
                             estaciones_vacias, primera_hora, ultima_hora])

control = pd.DataFrame(control_archivos, columns=['archivo', 'filas', 'fechas_invalidas',
                                                 'estaciones_vacias', 'primera_hora', 'ultima_hora'])
display(control)


## 3. Unir los resúmenes sin volver a leer los datos crudos

In [ ]:
if len(resumenes) == 0:
    raise ValueError('No se encontraron validaciones troncales con fecha y estación válidas.')

validaciones = pd.concat(resumenes, ignore_index=True)
validaciones = validaciones.groupby(['fecha', 'hora', 'Estacion_Parada'], as_index=False)['validaciones'].sum()
print('Registros agregados por día, hora y estación:', len(validaciones))
display(validaciones.head())


## 4. ¿Cuántas entradas se registraron por día y hora?

Antes de comparar días, revisa en la tabla `control` si sus primeras y últimas horas son parecidas. Si un archivo cubre solo parte del día, su total no es comparable directamente.

In [ ]:
por_dia = validaciones.groupby('fecha', as_index=False)['validaciones'].sum()
por_hora = validaciones.groupby(['fecha', 'hora'], as_index=False)['validaciones'].sum()
display(por_dia)
display(por_hora.head(20))

if plt is not None:
    tabla_horaria = por_hora.pivot(index='hora', columns='fecha', values='validaciones')
    tabla_horaria.plot(figsize=(10, 5), marker='o')
    plt.title('Entradas registradas por hora y día')
    plt.xlabel('Hora del día')
    plt.ylabel('Número de validaciones')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


## 5. ¿En qué estaciones se concentran más entradas?

La tabla y la gráfica reúnen todos los días disponibles. Esto muestra **volumen registrado**, no hacinamiento ni calidad del servicio. Los nombres de estación se conservan tal como vienen en la fuente; antes de unirlos con GTFS habrá que comprobar equivalencias.

In [ ]:
por_estacion = validaciones.groupby('Estacion_Parada', as_index=False)['validaciones'].sum()
por_estacion = por_estacion.sort_values('validaciones', ascending=False)
display(por_estacion.head(15))

if plt is not None:
    primeras_estaciones = por_estacion.head(10).sort_values('validaciones')
    primeras_estaciones.plot.barh(x='Estacion_Parada', y='validaciones', figsize=(10, 6), legend=False)
    plt.title('Estaciones con más entradas registradas en los días disponibles')
    plt.xlabel('Número de validaciones')
    plt.ylabel('Estación')
    plt.tight_layout()
    plt.show()


## 6. Guardar una tabla exploratoria sin sobrescribir resultados

La tabla `validaciones` tiene una fila por **fecha, hora y estación**. Se guarda con las fechas de inicio y fin en el nombre. **No reemplaza** `validaciones_por_estacion_hora.csv`, que es un resultado histórico, ni `powerbi_tabla_integrada_provisional.csv`, que alimenta el tablero actual. No se modifica ningún archivo crudo.

In [ ]:
carpeta_salida = carpeta_proyecto / 'datos' / 'procesados'
carpeta_salida.mkdir(parents=True, exist_ok=True)
fecha_inicial = validaciones['fecha'].min()
fecha_final = validaciones['fecha'].max()
nombre_salida = f'validaciones_por_estacion_hora_{fecha_inicial}_{fecha_final}.csv'
ruta_salida = carpeta_salida / nombre_salida
if ruta_salida.exists():
    raise FileExistsError(f'Ya existe {ruta_salida}. No se sobrescribió el resultado anterior.')
validaciones.to_csv(ruta_salida, index=False)
print('Tabla procesada guardada en:', ruta_salida)


## 7. Conclusión provisional y próximos pasos

Después de ejecutar el notebook, describe las estaciones y horas que aparecen arriba y anota cualquier diferencia de cobertura entre ZIP. **No afirmes** que hay menos buses, impuntualidad o discriminación a partir de validaciones.

Para conocer el cruce y la limpieza que usa el tablero, consulta `docs/bitacora_limpieza.md` y los programas de `src/`. Allí se emparejan las mismas fechas, horas y estaciones. Este notebook sigue siendo un ejercicio exploratorio; no conviertas sus totales en una medida de puntualidad o de pasajeros únicos.
